# Apps from Models - PySCIPOpt

This notebook shows how to:

1. Create a decision model that solves a knapsack problem with the
   SCIP solver.
2. Run it locally.
3. Push it to a Nextmv Cloud Application.
4. Run it remotely.

Let’s dive right in! 🐰

# Dependencies

Install the necessary Python packages.



In [ ]:
%pip install pyscipopt==5.6.0
%pip install "nextmv[all]==0.32.0"

# Imports

Add the necessary imports.

In [ ]:
import os
import json
import time
from importlib.metadata import version

import nextmv
import nextmv.cloud
import pyscipopt

# 1. Create the decision model

Use SCIP to solve a classic MIP with the `nextmv.Model` class.

In [ ]:
class DecisionModel(nextmv.Model):
    def solve(self, input: nextmv.Input) -> nextmv.Output:
        """Solves the given problem and returns the solution."""

        start_time = time.time()

        # Create the model.
        model = pyscipopt.Model("knapsack")
        # Define variables for assignment of items to the knapsack.
        x = []
        for item in input.data["items"]:
            x.append(model.addVar(vtype="B", obj=item["value"]))
        # Define constraint respecting the weight capacity of the knapsack.
        model.addCons(
            sum(x[i] * item["weight"] for i, item in enumerate(input.data["items"])) <= input.data["weight_capacity"]
        )
        # Maximize the total value of the knapsack.
        model.setMaximize()
        nVars, nCons = model.getNVars(), model.getNConss()

        # Solve the model.
        model.setParam("limits/time", input.options.duration)
        model.setParam("display/verblevel", 0)  # suppress output to support clean json on stdout
        model.optimize()

        # Determine which items were chosen.
        chosen_items = [item for i, item in enumerate(input.data["items"]) if model.getVal(x[i]) > 0.5]

        # Prepare the output.
        input.options.version = version("pyscipopt")
        statistics = nextmv.Statistics(
            run=nextmv.RunStatistics(duration=time.time() - start_time),
            result=nextmv.ResultStatistics(
                value=model.getObjVal(),
                custom={
                    "status": str(model.getStatus()),
                    "variables": nVars,
                    "constraints": nCons,
                },
            ),
        )

        return nextmv.Output(
            options=input.options,
            solution={"items": chosen_items},
            statistics=statistics,
        )

# 2. Run the model locally

Define the options that the model needs.

In [4]:
options = nextmv.Options(
    nextmv.Option("duration", int, 30, "Max runtime duration (in seconds).", False),
)

Instantiate the model.

In [5]:
model = DecisionModel()

Define some sample input data.

In [6]:
sample_input = {
  "items": [
    {
      "id": "cat",
      "value": 100,
      "weight": 20
    },
    {
      "id": "dog",
      "value": 20,
      "weight": 45
    },
    {
      "id": "water",
      "value": 40,
      "weight": 2
    },
    {
      "id": "phone",
      "value": 6,
      "weight": 1
    },
    {
      "id": "book",
      "value": 63,
      "weight": 10
    },
    {
      "id": "rx",
      "value": 81,
      "weight": 1
    },
    {
      "id": "tablet",
      "value": 28,
      "weight": 8
    },
    {
      "id": "coat",
      "value": 44,
      "weight": 9
    },
    {
      "id": "laptop",
      "value": 51,
      "weight": 13
    },
    {
      "id": "keys",
      "value": 92,
      "weight": 1
    },
    {
      "id": "nuts",
      "value": 18,
      "weight": 4
    }
  ],
  "weight_capacity": 50
}

Run the model locally.

In [ ]:
input = nextmv.Input(data=sample_input, options=options)
output = model.solve(input)
print(json.dumps(output.solution, indent=2))

# 3. Push the model to Nextmv Cloud

Convert the model to an application, hence the workflow name "Apps from
Models". Push the application to Nextmv Cloud.

Every app is production-ready with a full-featured API.

In [ ]:
client = nextmv.cloud.Client(api_key=os.getenv("NEXTMV_API_KEY"))
application = nextmv.cloud.Application(client=client, id="apps-from-models-pyscipopt-sample")

model_configuration = nextmv.ModelConfiguration(
    name="pyscipopt_model",
    requirements=[
        "pyscipopt==5.6.0",
        "nextmv==0.32.0"
    ],
    options=options,
)
manifest = nextmv.cloud.Manifest.from_model_configuration(model_configuration)
application.push(
    manifest=manifest,
    model=model,
    model_configuration=model_configuration,
    verbose=True,
)

# 4. Run the model remotely

Execute an app run. This remote run produces an output that should be the same as the local run.

In [ ]:
result = application.new_run_with_result(input=sample_input, instance_id="devint")
print(json.dumps(result.output, indent=2))
